Evaluating our Retrieval pipleine with Golden set of 25 questions 

In [ ]:
%run rag_setup.py

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Setup ready. Chunks: 680


In [ ]:
import json

with open("../evaluation/golden_set.json") as f:
    golden = json.load(f)

hits = 0
refused_right = 0
answerable = 0
unanswerable = 0

for item in golden:
    q = item["question"]
    results = vector_store.similarity_search_with_relevance_scores(q, k=3)
    best = results[0][1]
    files = [d.metadata["file"] for d, s in results]

    if item["expected_source"] is None:
        unanswerable += 1
        ok = best < 0.45
        refused_right += ok
        print(("PASS" if ok else "FAIL"), "| refuse check |", q, "| best:", round(best, 2))
    else:
        answerable += 1
        ok = item["expected_source"] in files
        hits += ok
        print(("PASS" if ok else "FAIL"), "| retrieval |", q)

print("\nRetrieval hit rate:", hits, "/", answerable)
print("Refusal accuracy:", refused_right, "/", unanswerable)


PASS | retrieval | How do I share a page with a guest?
PASS | retrieval | How can I change my Notion plan?
PASS | retrieval | Where can I see and update my billing information?
PASS | retrieval | What is Notion AI?
PASS | retrieval | How do I import a CSV file into Notion?
PASS | retrieval | How do I export my workspace content?
PASS | retrieval | What are rollups in a Notion database?
PASS | retrieval | How do database automations work?
PASS | retrieval | How can I set a reminder in Notion?
PASS | retrieval | How do I search for a page in Notion?
PASS | retrieval | What keyboard shortcuts are available in Notion?
PASS | retrieval | How do I publish a page as a website?
PASS | retrieval | How do comments and mentions work in Notion?
PASS | retrieval | What is a wiki in Notion?
PASS | retrieval | How do I create a form in Notion?
PASS | retrieval | How do I let someone outside my company see one of my pages?
PASS | retrieval | I want to pay for a better plan, where do I do that?
FAIL | 

C:\Users\sahil\AppData\Local\Temp\ipykernel_22688\1375849238.py:13: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to 

In [ ]:
def retrieve_final(question):
    results = vector_store.similarity_search_with_relevance_scores(question, k=10)
    best = results[0][1]
    if best < 0.45:
        return best, []
    queries = make_queries(question)
    pool, seen = [], set()
    for q in queries:
        docs_q = vector_store.similarity_search(q, k=5) + bm25_retriever.invoke(q)[:5]
        for d in docs_q:
            if d.page_content not in seen:
                seen.add(d.page_content)
                pool.append(d)
    pairs = [(question, d.page_content) for d in pool]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(pool, scores), key=lambda x: x[1], reverse=True)
    return best, [d for d, s in ranked[:3]]


In [ ]:
hits = 0
refused_right = 0
answerable = 0
unanswerable = 0

for item in golden:
    q = item["question"]
    best, docs = retrieve_final(q)
    files = [d.metadata["file"] for d in docs]

    if item["expected_source"] is None:
        unanswerable += 1
        ok = best < 0.45
        refused_right += ok
        print(("PASS" if ok else "FAIL"), "| refuse check |", q, "| best:", round(best, 2))
    else:
        answerable += 1
        ok = item["expected_source"] in files
        hits += ok
        print(("PASS" if ok else "FAIL"), "| pipeline |", q)

print("\nPipeline hit rate:", hits, "/", answerable)
print("Refusal accuracy:", refused_right, "/", unanswerable)


PASS | pipeline | How do I share a page with a guest?
PASS | pipeline | How can I change my Notion plan?
FAIL | pipeline | Where can I see and update my billing information?
PASS | pipeline | What is Notion AI?
PASS | pipeline | How do I import a CSV file into Notion?
PASS | pipeline | How do I export my workspace content?
FAIL | pipeline | What are rollups in a Notion database?
PASS | pipeline | How do database automations work?
PASS | pipeline | How can I set a reminder in Notion?
PASS | pipeline | How do I search for a page in Notion?
PASS | pipeline | What keyboard shortcuts are available in Notion?
FAIL | pipeline | How do I publish a page as a website?
PASS | pipeline | How do comments and mentions work in Notion?
FAIL | pipeline | What is a wiki in Notion?
PASS | pipeline | How do I create a form in Notion?
PASS | pipeline | How do I let someone outside my company see one of my pages?
FAIL | pipeline | I want to pay for a better plan, where do I do that?
PASS | pipeline | Can I 

C:\Users\sahil\AppData\Local\Temp\ipykernel_22688\3058651645.py:2: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to n